# FLUJO 1 — Escenario 6: Covarianza no estacionaria
- Simulo los datos
- Analizo la data
- Construyo la data que le pasaré al modelo

Este notebook produce los artefactos que consume `psbp_fd_iteracion.m`
(MATLAB) y, después de él, el notebook `10_03_resultados`.

# 1. Imports y rutas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import sys
from pathlib import Path
from types import SimpleNamespace

# ── Generadores de datos y contrato de artefactos (pipelines) ────────────────
from model_psbp_fd.pipelines import (
    ConfigEscenario6, generar_escenario_6, guardar_escenario,
    guardar_curvas, guardar_representacion, guardar_fpca,
    guardar_estandarizador, guardar_datasets_ar,
    guardar_hiperparametros, guardar_config_evaluacion,
    verificar_contrato,
)

# ── Preprocesamiento funcional (functions_models) ────────────────────────────
from model_psbp_fd.functions_models import (
    FunctionalRepresentation, FPCA_L2, base_en_grilla, DataStandardizer,
)

# ── Evaluacion: lineas base sobre el bloque de prueba (fit) ──────────────────
from model_psbp_fd.fit import tabla_baselines

# ── Utilidades ───────────────────────────────────────────────────────────────
from model_psbp_fd.utils import get_project_root

# ── Visualizacion ────────────────────────────────────────────────────────────
from model_psbp_fd.graphics import (
    plot_empirical_sample, plot_functional_mean, plot_functional_variance,
    plot_mean_and_variance, plot_fts_empirical, plot_fts_functional,
    plot_scatter_theta, plot_functional_comparison,
    plot_diagnostico_estandarizacion, plot_fpca_scree, plot_seleccion_basis,
    plot_rezagos_heatmap,
)

plt.style.use("seaborn-v0_8-darkgrid")
%matplotlib inline

## 1.1 Constantes a modificar según experimento

In [ ]:
# Buscamos la raiz del proyecto
PROJECT_ROOT = get_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT : {PROJECT_ROOT}")

# Definicion de parametros del experimento
BASENAME      = "escenario"
ESCENARIO_ID  = 6                          # Algoritmo 6 del anexo (covarianza no estacionaria)
REPLICA_ID    = 1
SEED          = 41232
EXPERIMENT_ID = f"{BASENAME}_{ESCENARIO_ID}"
print(f"Experiment ID : {EXPERIMENT_ID}")
print(f"Escenario     : {ESCENARIO_ID}   Replica : {REPLICA_ID}")
print(f"Seed (base)   : {SEED}")

PROP_TRAIN   = 0.80
T0_PREVISTO  = 240

## 1.2 Construcción de rutas

In [ ]:
_REPORT_DIR   = PROJECT_ROOT / "reports"  / "simulaciones" / EXPERIMENT_ID
_ARTEFACT_DIR = PROJECT_ROOT / "artefact" / "simulaciones" / EXPERIMENT_ID

PATHS = {
    "raw":          PROJECT_ROOT / "data" / "simulaciones" / "raw" / EXPERIMENT_ID,
    "functional":   PROJECT_ROOT / "data" / "simulaciones" / "processed" / "functional" / EXPERIMENT_ID,
    "predict":      PROJECT_ROOT / "data" / "simulaciones" / "processed" / "predict" / EXPERIMENT_ID,
    "out_report":   _REPORT_DIR,
    "out_artefact": _ARTEFACT_DIR,
}
for name, path in PATHS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"  {name:12s} → {path}")

# 2. Simulación

In [ ]:
# CONSTANTES DE LA SIMULACION — Escenario 6 (Algoritmo 6 del anexo)
#
#     a_tj = phi_j a_{t-1,j} + e_tj,   e_tj ~ N(0, lambda_j(t) (1 - phi_j^2))
#
# SEGUNDO BLOQUE del anexo. Sobre la misma construccion del Algoritmo 5, el
# espectro VARIA con el periodo: lambda_j(t) interpola entre un espectro inicial
# y uno final que intercambia las dos primeras componentes, con quiebre abrupto
# en t* = 270. Como T0 = 240, el quiebre cae DENTRO del bloque de prueba: la base
# estimada sobre el entrenamiento deja de estar ordenada segun la variabilidad
# del periodo que se evalua.
#
# Los defaults de ConfigEscenario6 YA SON los del Cuadro tab:escenarios.

SIM_CFG = ConfigEscenario6(
    seed = SEED,
    R    = 20,   # ver nota sobre el numero de replicas mas abajo
)

for k, v in SIM_CFG.to_dict().items():
    print(f"  {k:<24}: {v}")
print()
print(f"  espectro inicial      : {np.array2string(SIM_CFG.espectro_inicial(), precision=4)}")
print(f"  espectro final        : {np.array2string(SIM_CFG.espectro_final(),   precision=4)}")
print(f"  modo / t_quiebre      : {SIM_CFG.modo} / {SIM_CFG.t_quiebre}")

## 2.1 Simulación — Escenario 6: covarianza no estacionaria (Algoritmo 6)

Segundo bloque del anexo. La ley condicional sigue siendo gaussiana y unimodal;
lo que se incumple es la **estacionariedad de segundo orden** sobre la que
descansa tratar la media y las autofunciones como cantidades conocidas,
estimadas una única vez sobre el bloque de entrenamiento.

El espectro final intercambia las dos primeras componentes, de modo que el
**conjunto** de varianzas no cambia —y por tanto la degradación no puede
atribuirse a que el bloque de prueba sea más variable— sino únicamente su
**orden**, que es exactamente el supuesto que sostiene el truncamiento.

Conviene precisar el alcance: ese supuesto **no** lo relaja el modelo dinámico,
que flexibiliza la forma de la ley condicional y no la estructura de covarianza
que sostiene la representación. La degradación esperada alcanza por igual a todo
método que opere sobre la misma reducción, de modo que el escenario **no
discrimina entre especificaciones dinámicas**: acota el alcance de la reducción.

**Sobre el número de réplicas.** Se genera con `R=20` y el flujo usa la réplica 0.
La ventana posterior al quiebre tiene solo $T-t^*=30$ períodos, de los cuales el
diagnóstico descarta además los de adaptación de la varianza; con una sola
réplica el reordenamiento no se separa del ruido Monte Carlo y
`reordenamiento_detectado` puede dar `False` **sin que el generador falle**.
`semillas_replicas` garantiza que la réplica 0 sea idéntica con independencia de
cuántas se generen.

In [ ]:
# ── Generación ───────────────────────────────────────────────────────────────
salida = generar_escenario_6(SIM_CFG)

# El flujo 1 analiza UNA réplica; el estudio Monte Carlo completo itera sobre R.
REPLICA_IDX = 0
X_raw = salida.observaciones[REPLICA_IDX]          # (T, L): matriz observada
T, G  = X_raw.shape

# Coeficientes y trayectoria del espectro: INOBSERVABLES, uso exclusivo de
# diagnostico. Nunca deben entrar al pipeline de estimacion.
COEFICIENTES_LATENTES = salida.internos["coeficientes"][REPLICA_IDX]   # (T, J)
TRAYECTORIA_ESPECTRO  = salida.internos["trayectoria_espectro"]        # (T, J)
ESPECTRO_INICIAL      = salida.internos["espectro_inicial"]
ESPECTRO_FINAL        = salida.internos["espectro_final"]

# Alias de compatibilidad: el resto del notebook usa `domain.grid`
domain = SimpleNamespace(grid=salida.grilla)

print(f"Datos simulados : {X_raw.shape}  →  T={T} curvas, G={G} puntos")
print("Control de calidad del generador:")
for k, v in salida.diagnostico.items():
    if isinstance(v, float):
        print(f"  {k:36s} = {v:.6g}")
    elif isinstance(v, list) and len(v) > 6:
        print(f"  {k:36s} = [{len(v)} valores]")
    else:
        print(f"  {k:36s} = {v}")

# ── Guardar configuración de simulación ──────────────────────────────────────
simulation_config = {
    "sim_params":    salida.config.to_dict(),
    "diagnostico":   {k: (v if not isinstance(v, np.ndarray) else v.tolist())
                      for k, v in salida.diagnostico.items()},
    "replica_idx":   REPLICA_IDX,
    "experiment_id": EXPERIMENT_ID,
    "seed":          SEED,
    "T":             int(T),
    "G":             int(G),
}
with open(PATHS["raw"] / "simulation_config.json", "w", encoding="utf-8") as _f:
    json.dump(simulation_config, _f, indent=2, ensure_ascii=False)

_npz = guardar_escenario(salida, str(PATHS["raw"] / "escenario_6"),
                         incluir_curvas=True, incluir_internos=True)

print(f"\n[raw] simulation_config.json            → {PATHS['raw']}")
print(f"[raw] escenario_6.npz (salida completa) → {_npz}")

In [ ]:
# ── Diagnóstico del quiebre de covarianza ───────────────────────────────────
# Izquierda: varianza empírica móvil de las dos componentes intercambiadas,
# promediada sobre réplicas. El cruce debe ocurrir en t* y NO antes.
# Derecha: espectro empírico antes y después del quiebre.
_A_todas = salida.internos["coeficientes"]          # (R, T, J)
_i, _j   = (int(SIM_CFG.indices_intercambio[0]), int(SIM_CFG.indices_intercambio[1]))
_tq      = int(SIM_CFG.t_quiebre)
_VENT    = 20                                        # ventana móvil
_T0_fig  = int(np.floor(PROP_TRAIN * T))             # corte train/test

_energia = (_A_todas ** 2).mean(axis=0)              # (T, J) promedio sobre réplicas
_ker = np.ones(_VENT) / _VENT
_mov_i = np.convolve(_energia[:, _i - 1], _ker, mode="valid")
_mov_j = np.convolve(_energia[:, _j - 1], _ker, mode="valid")
_tt = np.arange(_VENT, T + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(_tt, _mov_i, color="#3b7dd8", lw=1.8, label=f"componente {_i}")
axes[0].plot(_tt, _mov_j, color="#e07b39", lw=1.8, label=f"componente {_j}")
axes[0].axvline(_tq, color="#b5561f", ls="--", lw=1.6, label=f"$t^*$ = {_tq}")
axes[0].axvline(_T0_fig, color="0.45", ls=":", lw=1.6,
                label=f"$T_0$ = {_T0_fig}")
axes[0].set_xlabel("t"); axes[0].set_ylabel(f"varianza móvil ({_VENT} períodos)")
axes[0].set_title("Intercambio de varianza entre las dos primeras componentes")
axes[0].legend(fontsize=8)

_J   = ESPECTRO_INICIAL.size
_jj  = np.arange(1, _J + 1)
_anc = 0.38
axes[1].bar(_jj - _anc / 2, salida.diagnostico["var_empirica_pre"], _anc,
            label="empírica pre-quiebre", color="#3b7dd8")
axes[1].bar(_jj + _anc / 2, salida.diagnostico["var_empirica_post"], _anc,
            label="empírica post-quiebre", color="#e07b39")
axes[1].set_xlabel("componente $j$"); axes[1].set_ylabel("varianza marginal")
axes[1].set_title("Espectro empírico antes y después de $t^*$")
axes[1].set_xticks(_jj); axes[1].legend(fontsize=8)

fig.suptitle("Escenario 6 — reordenamiento del espectro en el bloque de prueba")
fig.tight_layout()
fig.savefig(PATHS["out_report"] / "00_diagnostico_quiebre.png", dpi=150,
            bbox_inches="tight")
plt.show()

print(f"orden pre-quiebre  : {salida.diagnostico['orden_componentes_pre']}")
print(f"orden post-quiebre : {salida.diagnostico['orden_componentes_post']}")
print(f"reordenamiento detectado / esperado : "
      f"{salida.diagnostico['reordenamiento_detectado']} / "
      f"{salida.diagnostico['reordenamiento_esperado']}")
print(f"períodos de adaptación de la varianza: {salida.diagnostico['periodos_adaptacion']}")
print(f"t* estimado por CUSUM : {salida.diagnostico['instante_quiebre_estimado']}"
      f"   (objetivo {_tq}, error {salida.diagnostico['instante_quiebre_error']})")

if not salida.diagnostico["reordenamiento_detectado"]:
    print("\n  [ATENCION] El reordenamiento no se detectó con este R.")
    print("  La ventana posterior a t* tiene solo T - t* = "
          f"{T - _tq} períodos y el estimador promedia sobre réplicas:")
    print("  con R bajo el ruido Monte Carlo domina. NO es un fallo del generador.")
    print("  Suba R (≥ 20) o lea `espectro_post_error_max` en lugar del orden.")

## 2.2 Visualización de los datos empíricos

In [ ]:
# ── Serie de tiempo funcional empírica ───────────────────────────────────────
highlight_idx = [0, 1, T // 2, T - 1]

fig = plot_fts_empirical(
    X_raw, domain.grid,
    highlight_idx   = highlight_idx,
    title           = f"Covarianza no estacionaria — {T} curvas empíricas (escala original)",
    separator_every = 5,
    save_path       = str(PATHS["out_report"] / "01_fts_empirica_raw.png"),
)
plt.show()

In [ ]:
# ── Muestra de curvas empíricas ──────────────────────────────────────────────
fig = plot_empirical_sample(
    X_raw, domain.grid,
    sample_idx = [0, 1, 2, T // 2, T - 1],
    title      = "Muestra de 5 curvas empíricas (escala original)",
    save_path  = str(PATHS["out_report"] / "02_muestra_empirica_raw.png"),
)
plt.show()

In [ ]:
# ── Media y varianza funcional ───────────────────────────────────────────────
fig = plot_mean_and_variance(
    X_raw, domain.grid,
    show_std1 = True,
    show_std2 = True,
    title     = "Media y varianza funcional — Covarianza no estacionaria (escala original)",
    save_path = str(PATHS["out_report"] / "03_media_varianza_raw.png"),
)
plt.show()

## 2.3 Persistencia de datos crudos

In [ ]:
X = X_raw   # alias para el resto del notebook

# guardar_curvas escribe X_curves.npy y domain_grid.npy en 'functional', que es
# donde el flujo de resultados los busca.
_p = guardar_curvas(PATHS, X, domain.grid)
print(f"[functional] X_curves.npy    {X.shape}")
print(f"[functional] domain_grid.npy {domain.grid.shape}")

## 2.4 Partición temporal de entrenamiento y prueba

La partición respeta el orden de la serie: todo objeto **estimado a partir de
los datos** (selección GCV de la base, FPCA, estandarizador) se ajusta
exclusivamente con el bloque inicial $\{1,\dots,T_0\}$ y se aplica al bloque de
prueba mediante `transform`. El bloque de prueba queda reservado para la
evaluación fuera de muestra.

In [ ]:
# ── Partición temporal (holdout) ─────────────────────────────────────────────
T0 = int(np.floor(PROP_TRAIN * T))         # nº de curvas de entrenamiento

assert 10 < T0 < T, f"T0={T0} fuera de rango para T={T}."

idx_train = np.arange(0, T0)
idx_test  = np.arange(T0, T)
X_train, X_test = X[idx_train], X[idx_test]

print("Partición temporal (holdout):")
print(f"  entrenamiento : t ∈ [1, {T0}]      →  {X_train.shape}")
print(f"  prueba        : t ∈ [{T0+1}, {T}]  →  {X_test.shape}")
print(f"  proporción    : {T0/T:.1%} / {1 - T0/T:.1%}")

# 3. Representación funcional B-spline

In [ ]:
# SELECCIÓN DE PARÁMETROS DE LA REPRESENTACIÓN FUNCIONAL (n_basis, order)
N_BASIS_RANGE = range(2, min(30, T0 // 2))   # [HOLDOUT] rango acotado por T0
ORDER_RANGE   = range(2, 5)
selection_records = []

for order_bs in ORDER_RANGE:
    for nb in N_BASIS_RANGE:
        if nb < order_bs:
            continue
        try:
            fr_tmp = FunctionalRepresentation(method="bspline", n_basis=nb, order=order_bs)
            TH_tmp = fr_tmp.fit_transform(X_train, domain.grid)   # [HOLDOUT] solo train
            X_rec  = fr_tmp.reconstruct(TH_tmp)

            L_i    = X_train.shape[1]
            sse_c  = np.sum((X_train - X_rec) ** 2, axis=1)
            ss_tot = np.sum((X_train - X_train.mean(axis=0, keepdims=True)) ** 2)
            vr     = 1.0 - sse_c.sum() / ss_tot
            rmse_c = np.sqrt(sse_c / L_i)
            df_k   = nb
            gcv_c  = (L_i * sse_c / (L_i - df_k) ** 2
                      if L_i > df_k else np.full_like(sse_c, np.nan))

            selection_records.append({
                "n_basis": nb, "order": order_bs, "var_retained": vr,
                "rmse_mean": rmse_c.mean(), "rmse_max": rmse_c.max(),
                "gcv_mean": float(np.mean(gcv_c)),
            })
        except Exception as e:
            print(f"  [SKIP] n_basis={nb}, order={order_bs}: {e}")

sel_df = pd.DataFrame(selection_records)

# ── Criterio: mínimo de GCV promedio sobre las curvas ────────────────────────
sel_valid = sel_df.dropna(subset=["gcv_mean"])
best_row  = sel_valid.loc[sel_valid["gcv_mean"].idxmin()]
nb_best   = int(best_row["n_basis"])
ord_best  = int(best_row["order"])

display(sel_df.style
    .format({"var_retained": "{:.4%}", "rmse_mean": "{:.6f}",
             "rmse_max": "{:.6f}", "gcv_mean": "{:.6f}"})
    .background_gradient(subset=["gcv_mean"],     cmap="YlOrRd_r")
    .background_gradient(subset=["var_retained"], cmap="YlGn")
    .background_gradient(subset=["rmse_mean"],    cmap="YlOrRd_r"))

print(f"\nSelección por GCV mínimo: n_basis={nb_best}, order={ord_best}, "
      f"GCV={best_row['gcv_mean']:.6f}")
print(f"Verificación descriptiva: var_retained={best_row['var_retained']:.4%}, "
      f"rmse_mean={best_row['rmse_mean']:.6f}, rmse_max={best_row['rmse_max']:.6f}")

In [ ]:
fig = plot_seleccion_basis(
    sel_df, nb_best, ord_best,
    save_path=str(PATHS["out_report"] / "08_seleccion_basis.png"),
)
plt.show()

## 3.1 Ajuste y visualización de la representación funcional

In [ ]:
# CONSTANTES ELEGIDAS PARA LA REPRESENTACIÓN FUNCIONAL
# La elección es una decisión del analista: el GCV la SUGIERE, no la fija.
# NOTA (Bloque 2): el proceso vive EXACTAMENTE en un espacio de dimension
# J = 10 (base de Fourier). La base B-spline debe tener suficientes
# funciones para resolver el armonico mas alto (k_max = 5); si el error de
# representacion domina, suba NB_ELEGIDO antes de culpar al modelo.
NB_ELEGIDO  = nb_best     # ← fíjalo TÚ si quieres apartarte del GCV
ORD_ELEGIDO = ord_best
print(f"Base elegida  : n_basis={NB_ELEGIDO}, order={ORD_ELEGIDO}")
print(f"Sugerido GCV  : n_basis={nb_best}, order={ord_best}"
      + ("   (coinciden)" if (NB_ELEGIDO, ORD_ELEGIDO) == (nb_best, ord_best)
         else "   ← DIFIERE de la sugerencia; justificar en la tesis"))

# center=False es imprescindible y por eso se declara de forma explícita. Con
# center=True, `reconstruct` es el mapa AFÍN Theta Phi^T + media en lugar del
# lineal, y la función media contamina la base recuperada por `base_en_grilla`,
# la matriz de Gram y las autofunciones.
fr = FunctionalRepresentation(method="bspline", n_basis=NB_ELEGIDO,
                              order=ORD_ELEGIDO, center=False)

# [HOLDOUT] Ajuste SOLO con entrenamiento; proyección de toda la serie.
fr.fit(X_train, domain.grid)
THETA       = fr.transform(X, domain.grid)            # (T, K)
THETA_train = THETA[idx_train]
print(f"THETA shape: {THETA.shape}  (train={THETA_train.shape[0]}, test={T - T0})")

fig = plot_fts_functional(
    X, domain.grid,
    fr              = fr,
    highlight_idx   = highlight_idx,
    title           = f"Covarianza no estacionaria — {T} curvas (repr. B-spline, n_basis={NB_ELEGIDO}, order={ORD_ELEGIDO})",
    separator_every = 5,
    save_path       = str(PATHS["out_report"] / "09_fts_funcional_bspline.png"),
)
plt.show()

guardar_representacion(PATHS, fr, THETA,
                       extra={"T0": int(T0), "prop_train": float(PROP_TRAIN),
                              "ajustado_en": "train", "center": bool(fr.center),
                              "n_basis": int(NB_ELEGIDO), "order": int(ORD_ELEGIDO),
                              "n_basis_gcv": int(nb_best), "order_gcv": int(ord_best)})
print("[functional] functional_representation.pkl + theta.csv + fr_config.json")

## 3.2 FPCA sobre la base seleccionada

In [ ]:
# FPCA GENERALIZADO en métrica L² (functions_models.FPCA_L2).
# Base B-spline NO ortonormal ⇒ Gram W = <φ_j,φ_k>_{L²} ≠ I, de modo que la
# descomposición correcta resuelve el problema propio generalizado
#     (W^{1/2} S_θ W^{1/2}) z = λ z,   con ‖ψ_m‖_{L²} = 1.
# [HOLDOUT] el ajuste emplea SOLO el bloque de entrenamiento.

Phi  = base_en_grilla(fr, THETA.shape[1])          # (G, K)
fpca = FPCA_L2().fit(THETA_train, Phi, domain.grid)

_ver = fpca.verificar(THETA_train, fr=fr)

print("Verificación FPCA_L2 (sobre entrenamiento):")
for _k, _v in _ver.items():
    if isinstance(_v, float):
        print(f"  {_k:34s} = {_v:.3e}")
    elif isinstance(_v, (list, tuple)):
        print(f"  {_k:34s} = {len(_v)} criterios")
    else:
        print(f"  {_k:34s} = {_v}")

_cond = _ver["cond_W"]
print(f"\ncond(W) = {_cond:.3e}", end="  ")
if _cond > 1e10:
    print("← MUY ALTO: reduzca n_basis o revise el solapamiento de la base")
elif _cond > 1e6:
    print("← alto: vigile las componentes de menor varianza")
else:
    print("(condicionamiento sano)")

assert _ver["todo_ok"], (
    "Las identidades del FPCA generalizado no se cumplen. Si falla "
    "`err_linealidad_reconstruct_rel`, revise que la representación se haya "
    "ajustado con center=False."
)

# ── Diagnóstico dinámico: AR(1) propio de cada componente (≠ varianza) ───────
# Varianza = representación; AR(1) = dinámica. Una FPC de varianza baja puede
# tener AR fuerte (útil para pronóstico) y una de varianza alta puede ser ruido
# temporal. Esta distinción es el eje del Escenario 5.
K = fpca.evals.size
_Bf = fpca.B_full
SCORES_all = (THETA_train - fpca.mu_theta) @ (fpca.W @ _Bf)   # (T0, K)
s0, s1 = SCORES_all[:-1], SCORES_all[1:]
ar1_own = (s1 * s0).sum(0) / np.clip((s0 * s0).sum(0), 1e-12, None)

evals, var_cum = fpca.evals, fpca.var_cum
VAR_TARGET  = 0.95
M_SUGGERIDO = fpca.seleccionar_M(VAR_TARGET)

fpca_tbl = pd.DataFrame({
    "componente": np.arange(1, K + 1),
    "autovalor":  evals,
    "var_ratio":  fpca.var_ratio,
    "var_acum":   var_cum,
    "ar1_propio": ar1_own,
})
display(fpca_tbl.head(min(15, K)).style.format(
    {"autovalor": "{:.4e}", "var_ratio": "{:.4%}", "var_acum": "{:.4%}",
     "ar1_propio": "{:+.3f}"})
    .background_gradient(subset=["var_ratio"], cmap="YlGn")
    .background_gradient(subset=["ar1_propio"], cmap="coolwarm", vmin=-1, vmax=1))
print(f"\nK B-spline disponibles : {K}")
print(f"SUGERENCIA (var ≥ {VAR_TARGET:.0%}) : M_SUGGERIDO = {M_SUGGERIDO}   "
      f"← solo referencia; fija M_FPCA en la celda 3.3")

fig = plot_fpca_scree(
    evals, var_cum, M_SUGGERIDO, var_target=VAR_TARGET,
    save_path=str(PATHS["out_report"] / "10_fpca_scree.png"),
)
plt.show()

## 3.3 Fijar el número de componentes retenidas

In [ ]:
M_FPCA = int(M_SUGGERIDO)   # ← nº de componentes FPCA a retener (lo fijas TÚ)

assert isinstance(M_FPCA, (int, np.integer)) and 1 <= M_FPCA <= fpca.evals.size, (
    f"M_FPCA debe ser entero en [1, {fpca.evals.size}]. Recibido: {M_FPCA!r}"
)
fpca.set_M(int(M_FPCA))
M_fpca = fpca.M

# ── Objetos derivados ────────────────────────────────────────────────────────
Psi_grid = fpca.Psi_grid          # (G, M) autofunciones ortonormales en L²
mu_grid  = fpca.mu_grid           # (G,)
W        = fpca.W                 # (K, K) Gram
B        = fpca.B                 # (K, M)
mu_theta = fpca.mu_theta          # (K,)

# [HOLDOUT] scores de TODA la serie proyectados sobre la base de entrenamiento
SCORES       = fpca.transform(THETA)          # (T, M)
SCORES_train = SCORES[idx_train]
SCORES_test  = SCORES[idx_test]

print(f"M_fpca = {M_fpca}   var. explicada = {fpca.var_cum[M_fpca-1]:.4%}")
print(f"[train] max|media ξ|  = {np.abs(SCORES_train.mean(0)).max():.2e}   (≈ 0)")
print(f"[test]  max|media ξ|  = {np.abs(SCORES_test.mean(0)).max():.3f}")
print(f"[test]  var ξ / λ     = "
      f"{np.array2string(SCORES_test.var(0, ddof=1) / fpca.lambdas, precision=3)}")

# La FPCA se persiste en §4.0, junto con los scores estandarizados, para que
# SCORES y SCORES_STD queden escritos en una única operación consistente.
print("\n[nota] la FPCA se persiste en §4.0, junto con los scores estandarizados.")

# 4. Construcción de datasets AR(p) sobre scores de FPCA

## 4.0 Estandarización de scores ξ 


In [ ]:
# [HOLDOUT] El ajuste emplea SOLO el bloque de entrenamiento. `etiqueta` y el
# registro interno `n_ajuste` convierten esa disciplina en una propiedad
# VERIFICABLE.
scores_standardizer = DataStandardizer(method="zscore_column", ddof=0)
scores_standardizer.fit(SCORES_train, etiqueta=f"train[1:{T0}]")

_chk_std = scores_standardizer.verificar_ajuste(T0)
print(f"[holdout] estandarizador ajustado con {_chk_std['n_ajuste']} filas "
      f"= T0 ({_chk_std['etiqueta_ajuste']})  →  ok={_chk_std['ajuste_ok']}")

SCORES_STD       = scores_standardizer.transform(SCORES)
SCORES_STD_train = SCORES_STD[idx_train]
SCORES_STD_test  = SCORES_STD[idx_test]

print(scores_standardizer.summary())
print(f"SCORES_STD : shape={SCORES_STD.shape}  (train={T0}, test={T - T0})")
print(f"  [train] max|media| = {np.abs(SCORES_STD_train.mean(0)).max():.2e}")
print(f"  [train] max|std-1| = {np.abs(SCORES_STD_train.std(0) - 1).max():.2e}")
print(f"  [test]  media      = {np.array2string(SCORES_STD_test.mean(0), precision=3)}")
print(f"  [test]  std        = {np.array2string(SCORES_STD_test.std(0),  precision=3)}")

# ── Persistencia (contrato de artefactos) ────────────────────────────────────
guardar_estandarizador(PATHS, scores_standardizer)
_res_fpca = guardar_fpca(PATHS, fpca, SCORES, SCORES_STD=SCORES_STD,
                         meta_extra={"T0": int(T0)})
print("[functional] artefactos FPCA + fpca_scores_std.csv + scores_standardizer/")
print(f"             cond_W persistido = {_res_fpca['meta']['cond_W']:.3e}")

fig = plot_diagnostico_estandarizacion(
    SCORES_train, SCORES_STD_train, np.arange(1, M_fpca + 1),
    labels=("Scores ξ (escala λ)", "Scores ξ estandarizados"),
    title="estadísticas por componente FPCA",
    save_path=str(PATHS["out_report"] / "04_diagnostico_estandarizacion_scores.png"),
)
fig.axes[0].set_xlabel("componente m"); fig.axes[1].set_xlabel("componente m")
plt.show()

In [ ]:
# [HOLDOUT] la elección de rezagos es una decisión de modelado: solo train
T_theta = SCORES_STD_train.shape[0]
K_total = SCORES_STD_train.shape[1]

N_LAGS_MAX = 3
N_LAGS_MAX = int(np.clip(N_LAGS_MAX, 1, T_theta - 2))

def _spearman_block(Y, X):
    """Spearman columna-a-columna vía rangos (pandas; sin scipy)."""
    Yr = pd.DataFrame(Y).rank().to_numpy()
    Xr = pd.DataFrame(X).rank().to_numpy()
    Yc = Yr - Yr.mean(0); Xc = Xr - Xr.mean(0)
    return (Yc.T @ Xc) / np.outer(np.sqrt((Yc**2).sum(0)), np.sqrt((Xc**2).sum(0)))

n_cov = K_total * N_LAGS_MAX
corr_pearson  = np.zeros((K_total, n_cov))
corr_spearman = np.zeros((K_total, n_cov))
col_labels = []
y_block = SCORES_STD_train[N_LAGS_MAX:, :]

for lag in range(1, N_LAGS_MAX + 1):
    x_block = SCORES_STD_train[N_LAGS_MAX - lag : T_theta - lag, :]
    sp = _spearman_block(y_block, x_block)
    for j in range(K_total):
        c = (lag - 1) * K_total + j
        for k in range(K_total):
            corr_pearson[k, c] = np.corrcoef(y_block[:, k], x_block[:, j])[0, 1]
        corr_spearman[:, c] = sp[:, j]
        col_labels.append(rf"$\xi_{{t-{lag},{j+1}}}$")

row_labels = [rf"$\xi_{{t,{k+1}}}$" for k in range(K_total)]
band  = 1.96 / np.sqrt(len(y_block))
VCLIP = 0.6

fig = plot_rezagos_heatmap(
    corr_pearson, col_labels, row_labels,
    title=f"Pearson — respuesta(t) vs lags 1..{N_LAGS_MAX}",
    n_lags_max=N_LAGS_MAX, K_total=K_total, band=band, vclip=VCLIP,
    save_path=str(PATHS["out_report"] / "12a_rezagos_pearson.png"),
)
plt.show()

fig = plot_rezagos_heatmap(
    corr_spearman, col_labels, row_labels,
    title=f"Spearman — respuesta(t) vs lags 1..{N_LAGS_MAX}",
    n_lags_max=N_LAGS_MAX, K_total=K_total, band=band, vclip=VCLIP,
    save_path=str(PATHS["out_report"] / "12b_rezagos_spearman.png"),
)
plt.show()

## 4.1 Selección de rezagos y dataset final

In [ ]:
N_LAGS  = 2
K_total = SCORES_STD.shape[1]
COMPONENT_IDX = list(range(K_total))

n_train_eff = T0 - N_LAGS            # respuestas en t = N_LAGS+1 … T0
n_test_eff  = T - T0                 # respuestas en t = T0+1 … T

assert T0 > N_LAGS, f"T0={T0} debe superar N_LAGS={N_LAGS}."
assert len(COMPONENT_IDX) > 0, "COMPONENT_IDX no puede estar vacío."
assert len(COMPONENT_IDX) == len(set(COMPONENT_IDX)), "COMPONENT_IDX repetidos."
assert all(0 <= i < K_total for i in COMPONENT_IDX), "Índices fuera de rango."

n_components = len(COMPONENT_IDX)

print(f"K_total disponibles : {K_total}  (índices 0 … {K_total - 1})")
print(f"T / T0              : {T} / {T0}")
print(f"N_LAGS              : {N_LAGS}")
print(f"n_train_eff         : {n_train_eff}")
print(f"n_test_eff          : {n_test_eff}")
print()
print(f"  {'k_modelo':>8}  {'idx_THETA':>10}  {'nombre_resp':>14}")
print(f"  {'─'*8}  {'─'*10}  {'─'*14}")
for k_model, idx in enumerate(COMPONENT_IDX):
    print(f"  {k_model:>8}  {idx:>10}  {'fpc_' + str(idx + 1):>14}")

In [ ]:
# ── Construcción de DataFrames AR(p): bloques de entrenamiento y prueba ──────
SCORES_sel = SCORES_STD[:, COMPONENT_IDX]

cov_names = [
    f"fpc_{COMPONENT_IDX[j] + 1}_lag{lag}"
    for lag in range(1, N_LAGS + 1)
    for j in range(n_components)
]

def _dataset_bloque(k: int, t_ini: int, t_fin: int) -> pd.DataFrame:
    """
    Filas con respuesta en t ∈ [t_ini, t_fin) y predictores en t-1 … t-N_LAGS.

    Los rezagos del primer origen de prueba provienen del final del bloque de
    entrenamiento: son observaciones pasadas disponibles en cada origen, de modo
    que su uso es el condicionamiento que prescribe §2.2.3.1 (predicción a
    horizonte h=1 con la historia observada), no fuga de información.
    """
    t_idx  = np.arange(t_ini, t_fin)
    y_col  = SCORES_sel[t_idx, k]
    X_cols = np.hstack([SCORES_sel[t_idx - lag, :] for lag in range(1, N_LAGS + 1)])
    return pd.DataFrame(np.column_stack([y_col, X_cols]),
                        columns=[f"fpc_{COMPONENT_IDX[k] + 1}"] + cov_names)

dfs_train, dfs_test = {}, {}
for k in range(n_components):
    dfs_train[k] = _dataset_bloque(k, N_LAGS, T0)
    dfs_test[k]  = _dataset_bloque(k, T0,     T)

manifest = {
    "scores_scale":  "standardized_zscore_ddof0",
    "n_components":  n_components,
    "n_lags":        int(N_LAGS),
    "component_idx": [int(i) for i in COMPONENT_IDX],
    "cov_names":     cov_names,
    "T":             int(T),
    "T0":            int(T0),
    "prop_train":    float(PROP_TRAIN),
    "n_train_eff":   int(n_train_eff),
    "n_test_eff":    int(n_test_eff),
    "ajuste_en":     "train",
}
guardar_datasets_ar(PATHS, dfs_train, dfs_test, manifest)
print(f"[functional] {2*n_components} datasets (train/test) + datasets_manifest.json")

# 5. Especificación de hiperparámetros y ajuste MCMC

In [ ]:
MCMC_CONFIG = {"nsim": 2000, "burn": 500, "N": 20, "M": 50}
N_CHAINS    = 3
BURN        = int(MCMC_CONFIG["burn"])
print(f"MCMC_CONFIG : {MCMC_CONFIG}")
print(f"N_CHAINS    : {N_CHAINS}  (cadenas que ejecutará MATLAB)")
print()
print("Recordatorio: en `mcmc_config`, `M` es el tamaño de la grilla de")
print("localización G* del stick-breaking y `N` el truncamiento del número de")
print("átomos. NO confundir con M_FPCA (componentes retenidas).")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# HIPERPARÁMETROS — priors heterogéneas por tipo de variable
# ════════════════════════════════════════════════════════════════════════════

HP_GLOBAL = {
    "atau":  2.0,
    "btau":  0.5,
    "ag":    2.0,
    "bg":    0.5,
    "mumu":  0.0,
    "taumu": 1.0,
    "pwj":   0.5,
}

#   Formato: "tipo": (apij, bpij, mupsij, taupsij)
HP_BY_TYPE = {
    "own_lag1":  (9.0, 1.0,  0.0, 1.0),   # E[π] = 0.90
    "cross_lag": (1.0, 1.0,  0.0, 1.0),   # E[π] = 0.50
}

def _classify(name: str, k_model: int, component_idx: list) -> str:
    own_name = f"fpc_{component_idx[k_model] + 1}_lag1"
    if name == own_name:
        return "own_lag1"
    return "cross_lag"

HYPERPARAMS_LIST = []
for k in range(n_components):
    p = len(cov_names)
    apij    = np.empty(p); bpij    = np.empty(p)
    mupsij  = np.empty(p); taupsij = np.empty(p)

    for j, name in enumerate(cov_names):
        vtype         = _classify(name, k, COMPONENT_IDX)
        a, b, mu, tau = HP_BY_TYPE[vtype]
        apij[j]   = a;  bpij[j]    = b
        mupsij[j] = mu; taupsij[j] = tau

    HYPERPARAMS_LIST.append({**HP_GLOBAL,
                             "apij": apij,     "bpij": bpij,
                             "mupsij": mupsij, "taupsij": taupsij})

_W = 70
print("═" * _W)
print(f"  HYPERPARAMS_LIST  —  {n_components} componentes × {p} variables")
print("═" * _W)
print(f"  Globales: atau={HP_GLOBAL['atau']} btau={HP_GLOBAL['btau']}  "
      f"ag={HP_GLOBAL['ag']} bg={HP_GLOBAL['bg']}  "
      f"mumu={HP_GLOBAL['mumu']} taumu={HP_GLOBAL['taumu']}  "
      f"pwj={HP_GLOBAL['pwj']}")
print()
for k in range(n_components):
    hp = HYPERPARAMS_LIST[k]
    print(f"  Componente k={k+1}  (fpc_{COMPONENT_IDX[k]+1})")
    print(f"  {'Variable':<26} {'Tipo':<12} {'apij':>6} {'bpij':>6} "
          f"{'E[π]':>6} {'mupsij':>8} {'taupsij':>9}")
    print(f"  {'─'*26} {'─'*12} {'─'*6} {'─'*6} {'─'*6} {'─'*8} {'─'*9}")
    for j, name in enumerate(cov_names):
        vtype = _classify(name, k, COMPONENT_IDX)
        a  = hp["apij"][j];    b   = hp["bpij"][j]
        mu = hp["mupsij"][j];  tau = hp["taupsij"][j]
        e_pi = a / (a + b)
        marker = "  ◄" if vtype == "own_lag1" else ""
        print(f"  {name:<26} {vtype:<12} {a:>6.1f} {b:>6.1f} "
              f"{e_pi:>6.3f} {mu:>8.1f} {tau:>9.1f}{marker}")
    print()

In [ ]:
# ── Guardar hiperparámetros del modelo en artefact ───────────────────────────
hp_artifact = {
    "global":       HP_GLOBAL,
    "by_type":      HP_BY_TYPE,
    "mcmc_config":  MCMC_CONFIG,
    "n_iter":       N_CHAINS,
    "seed_scheme":  "SEED_BASE + chain*9973 + k*31",
    "escenario_id": int(ESCENARIO_ID),
    "replica_id":   int(REPLICA_ID),
    "seed_base":    SEED,
    "scores_scale": "standardized_zscore_ddof0",
    # [HOLDOUT] partición temporal — MATLAB entrena SOLO con *_train.csv
    "partition": {
        "T": int(T), "T0": int(T0), "prop_train": float(PROP_TRAIN),
        "n_train_eff": int(n_train_eff), "n_test_eff": int(n_test_eff),
        "train_files": [f"dataset_fpc_{COMPONENT_IDX[k]+1}_train.csv"
                        for k in range(n_components)],
        "test_files":  [f"dataset_fpc_{COMPONENT_IDX[k]+1}_test.csv"
                        for k in range(n_components)],
    },
    "hyperparams_list": []
}

for k in range(n_components):
    hp = HYPERPARAMS_LIST[k].copy()
    hp_artifact["hyperparams_list"].append({
        "component_k": k,
        "fpc_idx": int(COMPONENT_IDX[k] + 1),
        "hyperparams": {
            key: value.tolist() if isinstance(value, np.ndarray) else value
            for key, value in hp.items()
        }
    })

guardar_hiperparametros(PATHS, hp_artifact)
print(f"✓ Hiperparámetros guardados en: {PATHS['out_artefact'] / 'hyperparameters.json'}")
print()
print("SIGUIENTE PASO — MATLAB:")
print(f"  1. cd a notebooks/simulaciones/10_sim_E6")
print(f"  2. ejecutar psbp_fd_iteracion.m  (ESCENARIO_ID = 6 ya fijado)")
print(f"  3. volver a 10_03_resultados.ipynb")

# 6. Configuración de evaluación y líneas base (bloque de prueba)


In [ ]:
# ── Configuración de evaluación (la consume el flujo de resultados) ──────────
eval_config = {
    "scheme":         "holdout_temporal",
    "T":              int(T),
    "T0":             int(T0),
    "prop_train":     float(PROP_TRAIN),
    "horizons":       [1],                    # h=1 vía dataset_test; h>1: simulación
    "n_lags":         int(N_LAGS),
    "metrics_scores": ["RMSE", "R2"],
    # Error puntual en forma integrada (decisión cerrada del estudio):
    # MISE y MIAE son medias del error integrado por curva; `peor_curva` es su
    # máximo SOBRE CURVAS, no sobre puntos del dominio.
    "metrics_curvas": ["MISE", "MIAE", "RMSE_funcional", "peor_curva"],
    "metrics_dist":   ["LPS", "CRPS", "cobertura_95", "PIT"],
    # Dos objetivos de evaluación, ambos SIN ruido de medición:
    #   proyectada = fpca.reconstruct(scores observados)  → modo_residuo="ninguno"
    #   representacion = fr.reconstruct(theta observados)
    "objetivos_evaluacion": ["proyectada", "representacion"],
    "modo_residuo":   "ninguno",
    "scores_scale":   "standardized_zscore_ddof0",
}
guardar_config_evaluacion(PATHS, eval_config)
print("[out_artefact] eval_config.json")

# ── Líneas base sobre el bloque de prueba (h=1) ──────────────────────────────
baselines_df = tabla_baselines(
    SCORES_STD, T0,
    estandarizador=scores_standardizer,
    fpca=fpca,
    X_obs=X,
    tau=domain.grid,
    h=1,
)
baselines_df.to_csv(PATHS["out_report"] / "32_baselines_test.csv")
print("[out_report] 32_baselines_test.csv")
display(baselines_df.style.format("{:.4f}", na_rep="—")
        .set_caption("Líneas base — bloque de prueba, h=1 "
                     "(piso que el PSBP-FD debe superar)"))

# ── Verificación cruzada del contrato de artefactos ──────────────────────────
informe = verificar_contrato(PATHS)
print(f"\ncontrato_ok = {informe['contrato_ok']}  "
      f"(M={informe['M']}, K={informe['K']}, T0={informe['T0']}, "
      f"n_components={informe['n_components']})")
print(f"estandarizador ajustado con {informe.get('estandarizador_n_ajuste')} filas "
      f"(T0 = {informe['T0']})")
print(f"verificación FPCA: todo_ok = {informe['verificacion_fpca']['todo_ok']}")
assert informe["contrato_ok"], "El contrato de artefactos NO es consistente."
print("\n✓ Artefactos listos para MATLAB.")